<a href="https://colab.research.google.com/github/daniivelascoo/ifp-programacion-ia/blob/main/Teoria_2_3_B_Encoding_Avanzado_Student.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🧠 Teoría Bloque B: Traduciendo Texto a Números (Encoding)
**Sprint 2.3 - Ingeniería de Datos Avanzada**

Las Redes Neuronales y los algoritmos de ML son calculadoras gigantes. Solo entienden números. No saben qué es "Rojo" o "XL".
Tenemos que traducir (codificar) estas categorías. Pero, ¿cómo?

### 🎯 Objetivos:
1.  **Datos Ordinales:** Cuando el orden importa (Bajo < Medio < Alto) $\to$ `OrdinalEncoder`.
2.  **Datos Nominales:** Cuando no hay jerarquía (Rojo vs Azul) $\to$ `OneHotEncoder`.
3.  **El peligro del Label Encoding:** Por qué no debemos asignar números arbitrarios (1, 2, 3) a colores.

## 1. El Dataset: Una Tienda de Ropa

Vamos a crear un pequeño DataFrame con dos tipos de variables categóricas.
*   **Talla:** S, M, L (Tienen un orden lógico: S < M < L).
*   **Color:** Rojo, Azul, Verde (No tienen orden lógico).

In [1]:
import pandas as pd

df = pd.DataFrame({
    'talla': ['S', 'L', 'M', 'L', 'S'],
    'color': ['Rojo', 'Azul', 'Verde', 'Rojo', 'Azul'],
    'precio': [10, 20, 15, 20, 10]
})

print("--- Datos Originales ---")
display(df)

--- Datos Originales ---


,talla,color,precio
0,S,Rojo,10
1,L,Azul,20
2,M,Verde,15
3,L,Rojo,20
4,S,Azul,10


## 2. Ordinal Encoding (Para jerarquías)

Para la columna **Talla**, tiene sentido asignar números crecientes:
*   S $\to$ 0
*   M $\to$ 1
*   L $\to$ 2

La IA entenderá que 2 es mayor que 0 (L > S).
Usamos `OrdinalEncoder` de Scikit-Learn.

In [2]:
from sklearn.preprocessing import OrdinalEncoder

# 1. DEFINIR EL ORDEN
# Si no lo definimos, sklearn ordenará alfabéticamente (L, M, S) -> ¡ERROR!
# Tenemos que decirle explícitamente cuál es el orden correcto.
orden_tallas = [['S', 'M', 'L']]

# 2. INSTANCIAR
# Pasamos la lista de categorías
encoder_ord = OrdinalEncoder(categories=orden_tallas) # TODO: Pasa la variable orden_tallas

# 3. TRANSFORM
# Aplicamos la transformación a la columna 'talla'
# Ojo: Sklearn espera 2D (doble corchete)
tallas_encoded = encoder_ord.fit_transform(df[['talla']])

print("--- Tallas Codificadas (0=S, 1=M, 2=L) ---")
print(tallas_encoded)

--- Tallas Codificadas (0=S, 1=M, 2=L) ---
[[0.]
 [2.]
 [1.]
 [2.]
 [0.]]


## 3. El Peligro en Datos Nominales (Colores)

¿Qué pasa si usamos la técnica anterior para el **Color**?
*   Azul $\to$ 0
*   Rojo $\to$ 1
*   Verde $\to$ 2

**El Problema Matemático:**
La IA pensará que **Verde (2)** es "el doble" de bueno que **Azul (1)**. O que Rojo es el promedio entre Azul y Verde.
¡Esto es falso! Los colores no tienen rango. Esto confunde al modelo.

**Solución:** `OneHotEncoder`.

In [3]:
from sklearn.preprocessing import OneHotEncoder

# Creamos una columna binaria (0/1) para CADA color posible.
# ¿Es Rojo? Sí/No. ¿Es Azul? Sí/No.

# 1. INSTANCIAR
# sparse_output=False nos permite ver la matriz normal (no comprimida)
encoder_hot = OneHotEncoder(sparse_output=False) # TODO: Pon False

# 2. TRANSFORM
colores_encoded = encoder_hot.fit_transform(df[['color']])

# 3. VER RESULTADO CON NOMBRES
# Convertimos a DataFrame para entenderlo mejor
df_colores = pd.DataFrame(
    colores_encoded,
    columns=encoder_hot.get_feature_names_out(['color'])
)

print("\n--- One Hot Encoding (Sin jerarquía falsa) ---")
display(df_colores)

# Fíjate:
# Fila 0 (Rojo): [0, 1, 0] -> (No es Azul, Sí es Rojo, No es Verde)


--- One Hot Encoding (Sin jerarquía falsa) ---


,color_Azul,color_Rojo,color_Verde
0,0.0,1.0,0.0
1,1.0,0.0,0.0
2,0.0,0.0,1.0
3,0.0,1.0,0.0
4,1.0,0.0,0.0
